In [1]:
import os
import glob
import subprocess
import sys
from pathlib import Path

def run_contract_metrics_on_pattern(search_dir="../out", pattern="*test_outputs_8b.json"):
    """
    Automate running contract_level_metrics.py on multiple JSON files.
    
    Args:
        search_dir (str): Directory to search for JSON files
        pattern (str): File pattern to match (e.g., "*test_outputs_8b.json")
    """
    # Get the absolute path of the metrics script
    script_path = Path("contract_level_metrics.py").resolve()
    
    # Search for matching files
    search_pattern = os.path.join(search_dir, pattern)
    matching_files = glob.glob(search_pattern, recursive=True)
    
    if not matching_files:
        print(f"No files found matching pattern: {search_pattern}")
        return
    
    print(f"Found {len(matching_files)} files matching pattern '{pattern}' in '{search_dir}':")
    for file in matching_files:
        print(f"  - {file}")
    print("\n" + "="*80 + "\n")
    
    # Run metrics script on each file
    for i, json_file in enumerate(matching_files, 1):
        print(f"[{i}/{len(matching_files)}] Processing: {os.path.basename(json_file)}")
        print(f"Full path: {json_file}")
        
        try:
            # Run the contract_level_metrics.py script
            result = subprocess.run([
                sys.executable,  # Use current Python interpreter
                str(script_path),
                "--json_file_path", json_file
            ], capture_output=True, text=True, check=True)
            
            # Print the output
            if result.stdout:
                print("Output:")
                print(result.stdout)
            if result.stderr:
                print("Errors/Warnings:")
                print(result.stderr)
                
        except subprocess.CalledProcessError as e:
            print(f"Error running script on {json_file}:")
            print(f"Return code: {e.returncode}")
            print(f"stdout: {e.stdout}")
            print(f"stderr: {e.stderr}")
        except Exception as e:
            print(f"Unexpected error: {e}")
        
        # Add separator between runs
        print("\n" + "-"*60 + "\n")

# Example usage:
# Run on all files matching the pattern in the ../out directory
run_contract_metrics_on_pattern("../out", "*test_outputs_8b.json")


Found 5 files matching pattern '*test_outputs_8b.json' in '../out':
  - ../out/Intellectual_Property_&_Licensing_test_outputs_8b.json
  - ../out/Legal_Protections_&_Liability_test_outputs_8b.json
  - ../out/Termination_&_Control_Rights_test_outputs_8b.json
  - ../out/Competition_&_Exclusivity_test_outputs_8b.json
  - ../out/Financial_&_Commercial_Terms_test_outputs_8b.json


[1/5] Processing: Intellectual_Property_&_Licensing_test_outputs_8b.json
Full path: ../out/Intellectual_Property_&_Licensing_test_outputs_8b.json
Output:
Total instances in JSON file: 1386
Successfully parsed expected outputs: 1386/1386
Successfully parsed generated outputs: 1386/1386
Unique contracts found: 30

=== CONTRACT-LEVEL STATISTICS ===
Contracts with data: 30
Total expected clauses across all contracts: 174
Total generated clauses across all contracts: 186
Average expected clauses per contract: 5.80
Average generated clauses per contract: 6.20

=== OVERALL AGGREGATED METRICS ===
Total True Positives: 100


In [ ]:
import modal
import os
from pathlib import Path


# Create a local directory to save downloaded files
local_download_dir = Path("../src/stage1/out/base")
local_download_dir.mkdir(parents=True, exist_ok=True)

# Create a Modal app (previously called Stub)
app = modal.App("download-files")

# Reference the volume where your files are stored
volume = modal.Volume.from_name("eval-outputs")

@app.function(volumes={"/ckpts": volume})
def list_files_in_volume(path="/ckpts"):
    """List all files in the specified path in the volume."""
    import os
    files = []
    for root, dirs, filenames in os.walk(path):
        for filename in filenames:
            file_path = os.path.join(root, filename)
            files.append(file_path)
    return files

@app.function(volumes={"/ckpts": volume})
def download_file(file_path):
    """Download a specific file from the volume."""
    import os
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            return f.read()
    else:
        return None

def main():
    try:
        # First, deploy the app to Modal
        with app.run():
            # List all files in the volume
            files = list_files_in_volume.remote()
            print(f"Found {len(files)} files in the volume:")
            for file in files:
                print(f"  - {file}")
            
            # Download all files automatically
            print(f"\nDownloading all {len(files)} files...")
            
            for file_path in files:
                print(f"Downloading {file_path}...")
                file_content = download_file.remote(file_path)
                
                if file_content:
                    # Create local directory structure
                    relative_path = file_path.lstrip("/ckpts/").split("/")[-1]
                    local_file_path = local_download_dir / relative_path
                    local_file_path.parent.mkdir(parents=True, exist_ok=True)
                    
                    # Save the file
                    with open(local_file_path, "wb") as f:
                        f.write(file_content)
                    print(f"  Saved to {local_file_path}")
                else:
                    print(f"  Error: Could not download {file_path}")
            
            print(f"\nAll files downloaded to: {local_download_dir.absolute()}")
    except Exception as e:
        print(f"Error: {e}")

# Run the main function
main()


Found 6 files in the volume:
  - /ckpts/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-8B.json
  - /ckpts/Financial_&_Commercial_Terms_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Intellectual_Property_&_Licensing_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Legal_Protections_&_Liability_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Termination_&_Control_Rights_test_outputs_base_Qwen_Qwen3-1.7B.json

  Saved to ../src/stage1/out/base/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-8B.json
  Saved to ../src/stage1/out/base/Financial_&_Commercial_Terms_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Intellectual_Property_&_Licensing_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Legal_Protections_&_Liability_test_outputs_base_Qwen_Qwen3-

In [ ]:
import modal
import os
from pathlib import Path


# Create a local directory to save downloaded files
local_download_dir = Path("../src/stage1/out/base")
local_download_dir.mkdir(parents=True, exist_ok=True)

# Create a Modal app (previously called Stub)
app = modal.App("download-files")

# Reference the volume where your files are stored
volume = modal.Volume.from_name("eval-outputs")

@app.function(volumes={"/ckpts": volume})
def list_files_in_volume(path="/ckpts"):
    """List all files in the specified path in the volume."""
    import os
    files = []
    for root, dirs, filenames in os.walk(path):
        for filename in filenames:
            file_path = os.path.join(root, filename)
            files.append(file_path)
    return files

@app.function(volumes={"/ckpts": volume})
def download_file(file_path):
    """Download a specific file from the volume."""
    import os
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            return f.read()
    else:
        return None

def main():
    try:
        # First, deploy the app to Modal
        with app.run():
            # List all files in the volume
            files = list_files_in_volume.remote()
            print(f"Found {len(files)} files in the volume:")
            for file in files:
                print(f"  - {file}")
            
            # Download all files automatically
            print(f"\nDownloading all {len(files)} files...")
            
            for file_path in files:
                print(f"Downloading {file_path}...")
                file_content = download_file.remote(file_path)
                
                if file_content:
                    # Create local directory structure
                    relative_path = file_path.lstrip("/ckpts/").split("/")[-1]
                    local_file_path = local_download_dir / relative_path
                    local_file_path.parent.mkdir(parents=True, exist_ok=True)
                    
                    # Save the file
                    with open(local_file_path, "wb") as f:
                        f.write(file_content)
                    print(f"  Saved to {local_file_path}")
                else:
                    print(f"  Error: Could not download {file_path}")
            
            print(f"\nAll files downloaded to: {local_download_dir.absolute()}")
    except Exception as e:
        print(f"Error: {e}")

# Run the main function
main()


Found 6 files in the volume:
  - /ckpts/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-8B.json
  - /ckpts/Financial_&_Commercial_Terms_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Intellectual_Property_&_Licensing_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Legal_Protections_&_Liability_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Termination_&_Control_Rights_test_outputs_base_Qwen_Qwen3-1.7B.json

  Saved to ../src/stage1/out/base/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-8B.json
  Saved to ../src/stage1/out/base/Financial_&_Commercial_Terms_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Intellectual_Property_&_Licensing_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Legal_Protections_&_Liability_test_outputs_base_Qwen_Qwen3-

In [ ]:
import modal
import os
from pathlib import Path


# Create a local directory to save downloaded files
local_download_dir = Path("../src/stage1/out/base")
local_download_dir.mkdir(parents=True, exist_ok=True)

# Create a Modal app (previously called Stub)
app = modal.App("download-files")

# Reference the volume where your files are stored
volume = modal.Volume.from_name("eval-outputs")

@app.function(volumes={"/ckpts": volume})
def list_files_in_volume(path="/ckpts"):
    """List all files in the specified path in the volume."""
    import os
    files = []
    for root, dirs, filenames in os.walk(path):
        for filename in filenames:
            file_path = os.path.join(root, filename)
            files.append(file_path)
    return files

@app.function(volumes={"/ckpts": volume})
def download_file(file_path):
    """Download a specific file from the volume."""
    import os
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            return f.read()
    else:
        return None

def main():
    try:
        # First, deploy the app to Modal
        with app.run():
            # List all files in the volume
            files = list_files_in_volume.remote()
            print(f"Found {len(files)} files in the volume:")
            for file in files:
                print(f"  - {file}")
            
            # Download all files automatically
            print(f"\nDownloading all {len(files)} files...")
            
            for file_path in files:
                print(f"Downloading {file_path}...")
                file_content = download_file.remote(file_path)
                
                if file_content:
                    # Create local directory structure
                    relative_path = file_path.lstrip("/ckpts/").split("/")[-1]
                    local_file_path = local_download_dir / relative_path
                    local_file_path.parent.mkdir(parents=True, exist_ok=True)
                    
                    # Save the file
                    with open(local_file_path, "wb") as f:
                        f.write(file_content)
                    print(f"  Saved to {local_file_path}")
                else:
                    print(f"  Error: Could not download {file_path}")
            
            print(f"\nAll files downloaded to: {local_download_dir.absolute()}")
    except Exception as e:
        print(f"Error: {e}")

# Run the main function
main()


Found 6 files in the volume:
  - /ckpts/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-8B.json
  - /ckpts/Financial_&_Commercial_Terms_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Intellectual_Property_&_Licensing_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Legal_Protections_&_Liability_test_outputs_base_Qwen_Qwen3-1.7B.json
  - /ckpts/Termination_&_Control_Rights_test_outputs_base_Qwen_Qwen3-1.7B.json

  Saved to ../src/stage1/out/base/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Competition_&_Exclusivity_test_outputs_base_Qwen_Qwen3-8B.json
  Saved to ../src/stage1/out/base/Financial_&_Commercial_Terms_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Intellectual_Property_&_Licensing_test_outputs_base_Qwen_Qwen3-1.7B.json
  Saved to ../src/stage1/out/base/Legal_Protections_&_Liability_test_outputs_base_Qwen_Qwen3-

In [ ]:
# Quick and simple version for immediate testing
# Modify the path and pattern as needed

import os
import subprocess
import glob

# Configuration
SEARCH_DIR = "../out"  # Change this to your target directory
PATTERN = "*test_outputs_8b.json"  # Change this to your pattern
SCRIPT_PATH = "contract_level_metrics.py"

# Find and process files
files = glob.glob(os.path.join(SEARCH_DIR, PATTERN))
print(f"Found {len(files)} matching files")

for i, file_path in enumerate(files, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(files)}] Processing: {os.path.basename(file_path)}")
    print('='*60)
    
    # Run the script
    result = subprocess.run([
        "python", SCRIPT_PATH, 
        "--json_file_path", file_path
    ], capture_output=True, text=True)
    
    print(f"Output:\n{result.stdout}")
    if result.stderr:
        print(f"Errors:\n{result.stderr}")
